In [26]:
import pandas as pd
import re
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from transformers import BertTokenizer, BertForSequenceClassification
from fastapi import FastAPI

# Load Dataset
def load_data():
    df = pd.read_csv('Dataset_5971.csv')
    df.dropna(inplace=True)
    return df

# Preprocessing Function
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text

# Feature Engineering
def extract_features(df):
    df['Text'] = df['TEXT'].apply(preprocess_text)
    df['has_email'] = df['EMAIL'].apply(lambda x: 1 if x.lower() == 'yes' else 0)
    df['has_phone'] = df['PHONE'].apply(lambda x: 1 if x.lower() == 'yes' else 0)
    df['has_url'] = df['URL'].apply(lambda x: 1 if x.lower() == 'yes' else 0)
    return df

# Vectorizing Text Data
tfidf = TfidfVectorizer(max_features=10000)

def preprocess_data(df):
    X_text = tfidf.fit_transform(df['TEXT']).toarray()
    X_features = df[['has_email', 'has_phone', 'has_url']].values
    X = np.hstack((X_text, X_features))    
    df['LABEL'] = df['LABEL'].str.strip().str.lower()
    y = df['LABEL'].map({'ham': 0, 'spam': 1, 'smishing': 2})
    return train_test_split(X, y, test_size=0.2, random_state=42)

# Train ML Model
def train_ml_model(X_train, y_train):
    model = RandomForestClassifier(n_estimators=100)
    model.fit(X_train, y_train)
    return model

# Deep Learning Model (BERT)
class SpamDetector(nn.Module):
    def __init__(self):
        super(SpamDetector, self).__init__()
        self.bert = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3)
    
    def forward(self, input_ids, attention_mask):
        return self.bert(input_ids, attention_mask=attention_mask)

# API for Real-Time Detection


In [27]:
import nest_asyncio
import uvicorn
from pydantic import BaseModel

# Apply nest_asyncio to allow running FastAPI in Jupyter Notebook
nest_asyncio.apply()
app = FastAPI()

class MessageRequest(BaseModel):
    TEXT: str
    EMAIL: str
    PHONE: str
    URL: str

@app.post("/detect")
def detect_message(request: MessageRequest):
    features = extract_features(pd.DataFrame({'TEXT': [request.TEXT], 'EMAIL': [request.EMAIL], 'PHONE': [request.PHONE], 'URL': [request.URL]}))
    X_text = tfidf.transform([request.TEXT]).toarray()
    X_features = features[['has_email', 'has_phone', 'has_url']].values
    X = np.hstack((X_text, X_features))
    prediction = ml_model.predict(X)[0]
    label_map = {0: "ham", 1: "spam", 2: "Smishing"}
    return {"content": request.TEXT, "prediction": label_map[prediction]}

if __name__ == "__main__":
    df = load_data()
    df = extract_features(df)
    X_train, X_test, y_train, y_test = preprocess_data(df)
    ml_model = train_ml_model(X_train, y_train)
    uvicorn.run(app, host="127.0.0.1", port=8000)
    print("Model trained with accuracy:", accuracy_score(y_test, ml_model.predict(X_test)))

INFO:     Started server process [7748]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:65065 - "POST /detect HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [7748]


Model trained with accuracy: 0.9531380753138076
